# CALM-VAD — all datasets, one notebook (Colab)

Covers all 4 completed benchmarks **plus** optional HR-Avenue/HR-ShanghaiTech/
HR-UBnormal. Three ways data gets in, all automatic except one:

| dataset | how | manual step? |
|---|---|---|
| ShanghaiTech, UBnormal | single Drive file (STG-NF) | no |
| CHAD | single Drive file, native pose | no |
| Avenue | direct URLs (anomalib-verified) + pose extraction | no (GPU speeds it up) |
| HR-Avenue / HR-ShanghaiTech / HR-UBnormal | MoCoDAD Drive **folder** (>50 files) | **yes, once** |

**Set `Runtime -> Change runtime type -> T4 GPU`** before running — it's not
required (everything falls back to CPU) but makes Avenue's pose extraction
much faster.

Run cells top to bottom. Edit `RUN` in Cell 2 to pick which datasets.

## Cell 1 — install + code

In [ ]:
!pip -q install numpy scipy scikit-learn pyyaml gdown ultralytics opencv-python-headless
import os, sys, subprocess
sh = lambda cmd: subprocess.run(cmd, shell=True, check=False)

REPO_URL = 'https://github.com/FaizanAbbas512/Sentrix.git'   # '' to upload a zip instead
if REPO_URL:
    sh('rm -rf /content/sentrix && git clone --depth 1 %s /content/sentrix' % REPO_URL)
else:
    from google.colab import files
    up = files.upload(); z = next(iter(up))
    sh('rm -rf /content/sentrix && mkdir -p /content/sentrix && unzip -q "%s" -d /content/sentrix' % z)
    subs = [d for d in os.listdir('/content/sentrix') if os.path.isdir(f'/content/sentrix/{d}')]
    if 'calm' not in subs and len(subs) == 1:
        sh(f'cp -r /content/sentrix/{subs[0]}/* /content/sentrix/')

os.chdir('/content/sentrix'); sys.path.insert(0, '/content/sentrix')
assert os.path.isdir('calm'), 'calm/ not found - check REPO_URL or the uploaded zip'
for d in ('data/pose', 'data/raw', 'results'): os.makedirs(d, exist_ok=True)
sh('python -m calm.selftest | tail -3')

import torch
DEVICE = '0' if torch.cuda.is_available() else 'cpu'
print('pose-extraction device:', DEVICE)

## Cell 2 — dataset registry

`kind`: `auto` (single Drive file, `--auto` detects the layout —
ShanghaiTech/UBnormal/CHAD all use this), `avenue` (direct URLs + pose
extraction, built in), `manual_zip` (MoCoDAD HR poses — you fetch the zip
by hand once, see the markdown in Cell 3).

In [ ]:
DATASETS = {
  'shanghaitech': dict(kind='auto', gdrive_file='1o9h3Kh6zovW4FIHpNBGnYIRSbGCu-qPt', fps=24),
  'ubnormal':     dict(kind='auto', gdrive_file='1o9h3Kh6zovW4FIHpNBGnYIRSbGCu-qPt', fps=30),  # same zip
  'chad':         dict(kind='auto', gdrive_file='1g01Ay86cuPmKGsrfxddxwA1uG_7bFFmu', fps=30),
  'avenue':       dict(kind='avenue', fps=25),
  'hr_avenue':       dict(kind='manual_zip', pattern='*venue*.zip',   fps=25),
  'hr_shanghaitech': dict(kind='manual_zip', pattern='*hanghai*.zip', fps=24),
  'hr_ubnormal':     dict(kind='manual_zip', pattern='*bnormal*.zip', fps=30),
}

RUN = ['shanghaitech', 'ubnormal', 'chad', 'avenue']   # <- edit; add hr_* once their zip is ready

# custom link?  DATASETS['shanghaitech']['gdrive_file'] = 'NEW_ID_OR_LINK'

## Cell 3 — helpers

For `hr_avenue` / `hr_shanghaitech` / `hr_ubnormal` only: open
<https://drive.google.com/drive/folders/1aUDiyi2FCc6nKTNuhMvpGG_zLZzMMc83>,
right-click the dataset folder (Avenue / ShanghaiTech / UBnormal) →
**Download** (Google zips it, `gdown` can't because it's >50 files), then
either put the zip in your Google Drive or keep it ready to upload — the
helper below finds it or asks you to upload it.

In [ ]:
import glob, zipfile, tarfile, fnmatch, gdown
_DRIVE_MOUNTED = [False]

def _extract(arc, dest):
    os.makedirs(dest, exist_ok=True)
    if zipfile.is_zipfile(arc):   zipfile.ZipFile(arc).extractall(dest)
    elif tarfile.is_tarfile(arc): tarfile.open(arc).extractall(dest)
    else: raise RuntimeError(f'{arc} is not a zip/tar')
    for p in glob.glob(f'{dest}/**/*', recursive=True):
        if p != arc and (zipfile.is_zipfile(p) or (p.endswith(('.tar','.tgz','.tar.gz')) and tarfile.is_tarfile(p))):
            try: _extract(p, os.path.dirname(p))
            except Exception: pass

def _find_zip(pattern):
    for base in ('/content', '/content/drive/MyDrive'):
        if base.startswith('/content/drive') and not _DRIVE_MOUNTED[0]:
            try:
                from google.colab import drive; drive.mount('/content/drive'); _DRIVE_MOUNTED[0] = True
            except Exception: continue
        for p in glob.glob(f'{base}/**/*', recursive=True):
            if fnmatch.fnmatch(os.path.basename(p).lower(), pattern.lower()) and p.lower().endswith(('.zip','.tar','.tgz','.tar.gz')):
                return p
    return None

AVENUE_VIDEO_URL = 'http://www.cse.cuhk.edu.hk/leojia/projects/detectabnormal/Avenue_Dataset.zip'
AVENUE_GT_URL    = 'http://www.cse.cuhk.edu.hk/leojia/projects/detectabnormal/ground_truth_demo.zip'

def fetch(name, spec):
    dest = f'data/raw/{name}'
    if spec['kind'] == 'auto':
        if os.path.isdir(dest) and glob.glob(f'{dest}/**/*.npy', recursive=True):
            return dest
        g = spec['gdrive_file']
        src = g if 'http' in g else f'https://drive.google.com/uc?id={g}'
        gdown.download(src, f'data/raw/{name}.zip', quiet=False, fuzzy=True)
        _extract(f'data/raw/{name}.zip', dest)
        return dest
    if spec['kind'] == 'avenue':
        if os.path.exists(f'data/pose/{name}.json'):
            return None    # already extracted, run loop will use the cached pose json
        import urllib.request
        os.makedirs(dest, exist_ok=True)
        print('downloading Avenue videos...')
        urllib.request.urlretrieve(AVENUE_VIDEO_URL, f'{dest}/Avenue_Dataset.zip')
        print('downloading Avenue ground truth...')
        urllib.request.urlretrieve(AVENUE_GT_URL, f'{dest}/ground_truth_demo.zip')
        _extract(f'{dest}/Avenue_Dataset.zip', dest)
        _extract(f'{dest}/ground_truth_demo.zip', dest)
        from calm.datasets import convert_avenue_mat_gt
        convert_avenue_mat_gt(f'{dest}/ground_truth_demo/testing_label_mask', f'{dest}/gt_npy')
        return dest
    if spec['kind'] == 'manual_zip':
        z = _find_zip(spec['pattern'])
        if not z:
            from google.colab import files
            print(f"no zip matching {spec['pattern']} — upload it now:")
            up = files.upload(); z = os.path.join('/content', next(iter(up)))
        print('  using zip:', z)
        _extract(z, dest)
        return dest
    raise ValueError(spec['kind'])
print('helpers ready')

## Cell 4 — run every dataset in `RUN`

ShanghaiTech/UBnormal/CHAD go through `--auto` (layout auto-detect). Avenue
extracts poses once (cached to `data/pose/avenue.json`, GPU-accelerated if
available) then runs via `--generic`. Each produces
`results/calm_report_<name>.{json,txt}`.

In [ ]:
import os, subprocess
done = []
for name in RUN:
    spec = DATASETS[name]
    print('\n' + '=' * 62 + '\n  ' + name + '\n' + '=' * 62)
    try:
        root = fetch(name, spec)
    except Exception as e:
        print('  fetch failed:', e); continue

    if spec['kind'] == 'avenue':
        pose_json = f'data/pose/{name}.json'
        if not os.path.exists(pose_json):
            rc = subprocess.run(['python', '-m', 'calm.extract_poses',
                '--videos', f'{root}/Avenue Dataset/testing_videos',
                '--gt', f'{root}/gt_npy', '--out', pose_json, '--split', 'test',
                '--weights', 'yolo11n-pose.pt', '--imgsz', '480',
                '--device', DEVICE, '--stride', '1']).returncode
            if rc != 0:
                print('  !! pose extraction failed for avenue'); continue
        rc = subprocess.run(['python', '-m', 'calm.harness', '--generic', pose_json,
                             '--tag', name, '--fps', str(spec['fps'])]).returncode
    else:
        subprocess.run(['python', '-m', 'calm.datasets', root])          # show the tree
        rc = subprocess.run(['python', '-m', 'calm.harness', '--auto', root,
                             '--tag', name, '--fps', str(spec['fps']),
                             '--save-generic', f'data/pose/{name}.json']).returncode

    if rc == 0 and os.path.exists(f'results/calm_report_{name}.json'):
        done.append(name); print('  OK ->', f'results/calm_report_{name}.txt')
    else:
        print('  !! harness failed for', name, '— read the tree / errors above')
print('\nfinished:', done)

## Cell 5 — combined comparison table (all datasets, CALM-VAD vs baselines)

In [ ]:
import json, glob, os
rows = []
for jp in sorted(glob.glob('results/calm_report_*.json')):
    r = json.load(open(jp)); ds = r['tag']
    for s in r['streams']:
        rows.append(dict(dataset=ds, method=s['label'],
                         AUC=round(s['frame_auc'], 3),
                         eventF1=round(s['event']['f1_avg'], 3),
                         **{f"FAPH@{k.split('=')[-1]}": s[k] for k in s if k.startswith('faph@')}))
try:
    import pandas as pd
    df = pd.DataFrame(rows)
    from IPython.display import display; display(df)
except Exception:
    for x in rows: print(x)

print('\nCalibration (ECE raw -> cal) and cost per dataset:')
for jp in sorted(glob.glob('results/calm_report_*.json')):
    r = json.load(open(jp)); c_ = r['calibration']
    print(f"  {r['tag']:<16} ECE {c_['ece_raw']:.4f} -> {c_['ece_cal']:.4f}   "
          f"decision {r['cost']['decision_layer_ms_per_frame_mean']:.2f} ms/frame")

## Cell 6 — cross-dataset generalisation (every fit/test pair among `RUN`)
Uses the `data/pose/<name>.json` files saved in Cell 4.

In [ ]:
import json, itertools, os
ready = [n for n in RUN if os.path.exists(f'data/pose/{n}.json')]
for a, b in itertools.permutations(ready, 2):
    A = json.load(open(f'data/pose/{a}.json')); B = json.load(open(f'data/pose/{b}.json'))
    for x in A['clips']: x['split'] = 'calib'
    for x in B['clips']: x['split'] = 'test'
    out = f'data/pose/{a}__to__{b}.json'
    json.dump({'fps': B['fps'], 'clips': A['clips'] + B['clips']}, open(out, 'w'))
    rc = subprocess.run(['python', '-m', 'calm.harness', '--generic', out,
                         '--tag', f'{a}__to__{b}']).returncode
    if rc == 0:
        r = json.load(open(f'results/calm_report_{a}__to__{b}.json'))
        s = [x for x in r['streams'] if 'CALM' in x['label']][0]
        print(f'  {a:>13} -> {b:<13}  eventF1 {s["event"]["f1_avg"]:.3f}  AUC {s["frame_auc"]:.3f}  ECE_cal {r["calibration"]["ece_cal"]:.3f}')

## Cell 7 — download everything

In [ ]:
!zip -qr /content/calm_all.zip results data/pose/*.json
from google.colab import files; files.download('/content/calm_all.zip')
try:
    from google.colab import drive; drive.mount('/content/drive')
    !cp /content/calm_all.zip /content/drive/MyDrive/ && echo 'also copied to Drive/MyDrive'
except Exception as e:
    print('Drive copy skipped:', e)